In [6]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import io
import sys
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score

from typing import Tuple
from sklearn.base import RegressorMixin
from typing import Tuple, List
from sklearn.pipeline import Pipeline
from sklearn.base import RegressorMixin
from typing import Optional
from sklearn.base import ClassifierMixin

from xgboost import XGBClassifier
from sklearn.ensemble import StackingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

from typing import Tuple, List, Dict
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve, average_precision_score

from sklearn.pipeline import make_pipeline
import optuna.visualization as vis
from statsmodels.tsa.arima.model import ARIMA


from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

import textwrap, pathlib, json, sys, inspect, os, types, importlib, pathlib, matplotlib, pandas as pd, numpy as np




In [5]:
# -------------------------------------------------------------
#  run_ts_pipeline.py  |  copia el contenido en una celda .ipynb
# -------------------------------------------------------------
import warnings, pathlib, inspect
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
from sklearn.metrics        import mean_squared_error
from sklearn.ensemble       import RandomForestRegressor
from sklearn.preprocessing  import StandardScaler

# --- librerías de series temporales
from statsmodels.tsa.seasonal   import seasonal_decompose
import pmdarima as pm
from skforecast.ForecasterAutoreg       import ForecasterAutoreg
from skforecast.ForecasterAutoregCustom import ForecasterAutoregCustom
from skforecast.ForecasterRecursive     import ForecasterRecursive

def run_ts_pipeline(
        data_src,
        date_col:   str,
        target_col: str,
        method:     str = "decompose",
        freq:       str | None = None,
        steps:      int = 36,
        exog_cols:  list[str] | None = None,
        test_ratio: float = .2,
        random_state: int = 123,
        plot: bool = True,
        **method_kwargs):
    """
    Ejecuta, grafica y devuelve resultados de diferentes enfoques de series temporales.

    Parámetros
    ----------
    data_src : str | pd.DataFrame
        Ruta a un .csv/.parquet o un DataFrame ya cargado.
    date_col : str
        Nombre de la columna con la fecha.
    target_col : str
        Variable a modelar / descomponer / pronosticar.
    method : {'decompose', 'recursive_rf', 'auto_arima', 'sarima'}
        Estrategia a utilizar.
    freq : str | None
        Frecuencia pandas compatibles ('MS', 'D', '30min' …).  Si None se infiere.
    steps : int
        Horizonte de predicción (solo forecasting).
    exog_cols : list[str] | None
        Columnas exógenas para métodos que las soportan.
    test_ratio : float
        Proporción del set final que queda para test.
    random_state : int
        Semilla para reproducibilidad.
    plot : bool
        Si True, se generan gráficas dentro del notebook.
    **method_kwargs : dict
        Argumentos extra que se pasan al modelo concreto.

    Retorna
    -------
    dict
        Con llaves 'model', 'pred', 'train', 'test', 'fig' (si plot=True).
    """
    # ---------------- Cargar datos -----------------
    if isinstance(data_src, (str, pathlib.Path)):
        if str(data_src).lower().endswith(".parquet"):
            df = pd.read_parquet(data_src)
        else:
            df = pd.read_csv(data_src)
    elif isinstance(data_src, pd.DataFrame):
        df = data_src.copy()
    else:
        raise TypeError("data_src debe ser ruta o DataFrame")

    # --------------- Pre-procesado base ------------
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).set_index(date_col)

    if freq:
        df = df.asfreq(freq)

    y = df[target_col]

    exog = df[exog_cols] if exog_cols else None

    # ------------ Split train / test ---------------
    split = int(len(df) * (1 - test_ratio))
    y_train, y_test = y.iloc[:split], y.iloc[split:]
    exog_train = exog.iloc[:split] if exog is not None else None
    exog_test  = exog.iloc[split:]  if exog is not None else None

    figures = []

    # --------------- Método elegido ----------------
    if method == "decompose":
        # --- descomposición clásica aditiva ----
        result = seasonal_decompose(y, model='additive', extrapolate_trend='freq')
        if plot:
            fig = result.plot()
            fig.set_size_inches(8,6)
            figures.append(fig)
        return {"model": result, "fig": figures}

    elif method == "recursive_rf":
        # --- forecasting recursivo con RandomForest ----
        forecaster = ForecasterRecursive(
            regressor = RandomForestRegressor(random_state=random_state, **method_kwargs),
            lags      = method_kwargs.get("lags", 12)
        )
        forecaster.fit(y=y_train, exog=exog_train)
        pred = forecaster.predict(steps=steps, exog=exog_test)

        if plot:
            fig, ax = plt.subplots(figsize=(8,3))
            y_train.plot(ax=ax, label="train")
            y_test.plot(ax=ax,  label="test")
            pred.plot(ax=ax,    label="predicción")
            ax.legend(); figures.append(fig)

        print(f"MSE test: {mean_squared_error(y_test[:len(pred)], pred):.3f}")

        return {"model": forecaster, "pred": pred,
                "train": y_train, "test": y_test, "fig": figures}

    elif method == "auto_arima":
        model = pm.auto_arima(
                    y = y_train,
                    exogenous = exog_train,
                    seasonal  = method_kwargs.get("seasonal", False),
                    m         = method_kwargs.get("m", 1),
                    stepwise  = True,
                    suppress_warnings=True,
                    random_state=random_state,
                    **{k:v for k,v in method_kwargs.items()
                       if k not in ("seasonal","m")}
                )
        model.fit(y_train, exogenous=exog_train)
        pred, conf_int = model.predict(
                            n_periods=steps,
                            exogenous=exog_test,
                            return_conf_int=True)
        pred = pd.Series(pred, index=y_test.index[:steps])

        if plot:
            fig, ax = plt.subplots(figsize=(8,3))
            y_train.plot(ax=ax); y_test.plot(ax=ax)
            pred.plot(ax=ax, color="red", label="forecast")
            ax.fill_between(pred.index, conf_int[:,0], conf_int[:,1],
                            alpha=.2, color='orange')
            ax.legend(); figures.append(fig)

        print(f"MSE test: {mean_squared_error(y_test[:steps], pred):.3f}")

        return {"model": model, "pred": pred,
                "train": y_train, "test": y_test, "fig": figures}

    elif method == "sarima":
        model = pm.auto_arima(
                    y_train,
                    start_p=1, start_q=1,
                    seasonal=True, m=method_kwargs.get("m", 12),
                    start_P=0, D=1,
                    stepwise=True, suppress_warnings=True,
                    random_state=random_state,
                    trace=False
               )
        model.fit(y_train)
        pred, conf_int = model.predict(n_periods=steps, return_conf_int=True)
        pred = pd.Series(pred, index=y_test.index[:steps])

        if plot:
            fig, ax = plt.subplots(figsize=(8,3))
            y_train.plot(ax=ax); y_test.plot(ax=ax)
            pred.plot(ax=ax, color="red", label="forecast")
            ax.fill_between(pred.index, conf_int[:,0], conf_int[:,1],
                            alpha=.2, color='gray')
            ax.legend(); figures.append(fig)

        print(f"MSE test: {mean_squared_error(y_test[:steps], pred):.3f}")

        return {"model": model, "pred": pred,
                "train": y_train, "test": y_test, "fig": figures}

    else:
        raise ValueError(f"method '{method}' no implementado "
                         "(elige 'decompose', 'recursive_rf', 'auto_arima', 'sarima')")



ModuleNotFoundError: No module named 'skforecast.ForecasterAutoreg'

In [3]:
from run_ts_pipeline import run_ts_pipeline          # si lo guardas como módulo
# o simplemente copia la definición previa en una celda y ejecútala

# 1) Descomposición estacional de un CSV mensual
run_ts_pipeline(
    data_src   = "Datos_agregados.csv",
    date_col   = "fecha",
    target_col = "y",
    method     = "decompose",
    freq       = "MS"            # Monthly Start
)

# 2) Forecast 36 pasos con RandomForest recursivo
run_ts_pipeline(
    data_src   = "Datos_agregados.csv",
    date_col   = "fecha",
    target_col = "y",
    method     = "recursive_rf",
    steps      = 36,
    exog_cols  = ["temperatura_media", "presion"],   # opcional
    lags       = 24,        # se pasa vía **method_kwargs
    n_estimators = 300,     # idem
)

# 3) Auto-ARIMA sin estacionalidad
run_ts_pipeline(
    df, "fecha", "y",
    method = "auto_arima",
    steps  = 24,
    seasonal = False
)

# 4) SARIMA automático (m=12 meses)
run_ts_pipeline(
    df, "fecha", "y",
    method = "sarima",
    steps  = 24,
    m      = 12
)


ModuleNotFoundError: No module named 'ts_automator'